# OntologyQA - OpenRouter Smoke Test

Notebook này kiểm tra nhanh việc gọi OpenRouter cho baseline **1.1 LLM-only** bằng 1 mẫu từ `../test_questions_v1.0.xlsx`.

Mục tiêu của notebook:
- Đọc 1 câu hỏi trắc nghiệm hợp lệ từ file Excel.
- Gọi OpenRouter qua `openai` SDK.
- Yêu cầu model chọn đáp án, không dùng SPARQL/ontology retrieval.
- Ghi lại input/output tokens, cost nếu OpenRouter trả về, và round-trip latency.
- Lưu kết quả kèm `question_type` để sau này thống kê theo subset.

## 1. Cấu hình môi trường

Notebook nằm trong `notebooks/`, còn `.env` và `test_questions_v1.0.xlsx` nằm ở project root. Có thể chọn sample bằng biến `SAMPLE_ID` trong `.env`.

In [6]:
import csv
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "test_questions_v1.0.xlsx").exists() and (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy project root chứa test_questions_v1.0.xlsx và .env.")


PROJECT_ROOT = find_project_root()
EXCEL_PATH = PROJECT_ROOT / "test_questions_v1.0.xlsx"
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH)

METHOD_ID = "1.1"
METHOD_NAME = "LLM-only baseline"
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
MODEL = os.getenv("OPENROUTER_MODEL", "google/gemma-4-26b-a4b-it")
API_KEY = os.getenv("OPENROUTER_API_KEY")
SAMPLE_ID = 43
TEMPERATURE = 0
MAX_TOKENS = 256
RUN_ID = f"method-1-1-smoke-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:8]}"
RESULTS_DIR = PROJECT_ROOT / "results" / "method_1_1_smoke" / RUN_ID

print(f"Project root: {PROJECT_ROOT}")
print(f"Excel exists: {EXCEL_PATH.exists()} -> {EXCEL_PATH.resolve()}")
print(f".env exists: {ENV_PATH.exists()} -> {ENV_PATH.resolve()}")
print(f"OpenRouter base URL: {OPENROUTER_BASE_URL}")
print(f"Model: {MODEL}")
print(f"Sample ID: {SAMPLE_ID}")
print(f"Run ID: {RUN_ID}")
print(f"API key configured: {bool(API_KEY)}")

Project root: D:\Dev\VDT2026-OntologyQA
Excel exists: True -> D:\Dev\VDT2026-OntologyQA\test_questions_v1.0.xlsx
.env exists: True -> D:\Dev\VDT2026-OntologyQA\.env
OpenRouter base URL: https://openrouter.ai/api/v1
Model: google/gemma-4-26b-a4b-it
Sample ID: 43
Run ID: method-1-1-smoke-20260613-164159-aafdf3f0
API key configured: True


## 2. Đọc 1 mẫu hợp lệ từ Excel

File hiện có một số dòng trống/chưa hoàn thiện. Cell này chỉ lấy dòng có đủ câu hỏi, chỉ số đáp án đúng và ít nhất 2 option.

In [7]:
def none_if_nan(value):
    return None if pd.isna(value) else value


def normalize_question_record(record: dict) -> dict | None:
    question = none_if_nan(record.get("vi_question"))
    correct_option = none_if_nan(record.get("answer"))
    options = [none_if_nan(record.get(f"option_{idx}")) for idx in range(1, 6)]
    available_options = [option for option in options if option is not None]

    if not question or correct_option is None or len(available_options) < 2:
        return None

    return {
        "number": int(record.get("number")),
        "question": str(question),
        "question_type": str(none_if_nan(record.get("question_type")) or "").strip(),
        "gold_answer": none_if_nan(record.get("gold_answer")),
        "correct_option": int(correct_option),
        "options": options,
    }


def load_valid_questions(path: Path) -> list[dict]:
    df = pd.read_excel(path, engine="openpyxl")
    df = df.rename(columns={"Unnamed: 3": "gold_answer"})
    records = []

    for record in df.to_dict(orient="records"):
        normalized = normalize_question_record(record)
        if normalized is not None:
            records.append(normalized)

    return records


def get_question_by_id(path: Path, sample_id: int) -> dict:
    records = load_valid_questions(path)
    for record in records:
        if record["number"] == sample_id:
            return record

    available_ids = [record["number"] for record in records]
    raise ValueError(f"Không tìm thấy sample id {sample_id}. Các id hợp lệ: {available_ids}")


sample = get_question_by_id(EXCEL_PATH, SAMPLE_ID)
print(json.dumps(sample, ensure_ascii=False, indent=2))

{
  "number": 43,
  "question": "Kể tên vị trí của những nhà sản xuất máy bay đã chọn dòng Sukhoi Su-27 để phát triển tiếp các model máy bay khác?",
  "question_type": "multi-hop",
  "gold_answer": "Liaoning, Shenyang, Begoyoy District, Moscow, Russia",
  "correct_option": 4,
  "options": [
    "Liaoning, Shenyang, Vietnam",
    " Liaoning, Shenyang",
    " Begoyoy District, Moscow, Russia",
    "Liaoning, Shenyang, Begoyoy District, Moscow, Russia",
    "VIetnam, Liaoning, Shenyang, Begoyoy District, Moscow, Russia"
  ]
}


## 3. Prompt baseline 1.1

Baseline 1.1 không gọi SPARQL, không retrieve ontology, không dùng dữ liệu ngoài. Model chỉ dựa vào parametric memory và các option đã cho.

In [8]:
def build_llm_only_messages(item: dict) -> list[dict[str, str]]:
    options_text = "\n".join(
        f"{idx}. {option}"
        for idx, option in enumerate(item["options"], start=1)
        if option is not None
    )

    user_prompt = f"""Câu hỏi tiếng Việt:
{item['question']}

Loại câu hỏi: {item['question_type'] or 'unknown'}

Các lựa chọn:
{options_text}

Hãy chọn đúng 1 đáp án trong các lựa chọn trên.
Chỉ trả về JSON hợp lệ theo schema:
{{"selected_option": <số nguyên 1-5>, "answer_text": "<nội dung đáp án>", "reason": "<giải thích ngắn>"}}
"""

    return [
        {
            "role": "system",
            "content": (
                "Bạn là baseline LLM-only cho bài toán OntologyQA. "
                "Không sinh SPARQL, không gọi công cụ, không giả định có truy cập knowledge graph. "
                "Chỉ chọn đúng một đáp án từ các lựa chọn được cung cấp."
            ),
        },
        {"role": "user", "content": user_prompt},
    ]


messages = build_llm_only_messages(sample)
print(messages[1]["content"])

Câu hỏi tiếng Việt:
Kể tên vị trí của những nhà sản xuất máy bay đã chọn dòng Sukhoi Su-27 để phát triển tiếp các model máy bay khác?

Loại câu hỏi: multi-hop

Các lựa chọn:
1. Liaoning, Shenyang, Vietnam
2.  Liaoning, Shenyang
3.  Begoyoy District, Moscow, Russia
4. Liaoning, Shenyang, Begoyoy District, Moscow, Russia
5. VIetnam, Liaoning, Shenyang, Begoyoy District, Moscow, Russia

Hãy chọn đúng 1 đáp án trong các lựa chọn trên.
Chỉ trả về JSON hợp lệ theo schema:
{"selected_option": <số nguyên 1-5>, "answer_text": "<nội dung đáp án>", "reason": "<giải thích ngắn>"}



## 4. Gọi OpenRouter

Cell này gọi OpenRouter và đo latency theo round-trip time: từ lúc gửi request đến lúc nhận response xong ở client.

In [9]:
if not API_KEY:
    raise RuntimeError("Thiếu OPENROUTER_API_KEY. Hãy điền API key trong .env trước khi chạy cell này.")

client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=API_KEY,
)

started = time.perf_counter()
completion = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
)
round_trip_latency_ms = (time.perf_counter() - started) * 1000

completion_dict = completion.model_dump()
content = completion.choices[0].message.content or ""
usage = completion_dict.get("usage") or {}

print("Round-trip latency (ms):", round(round_trip_latency_ms, 2))
print("Model response:")
print(content)
print("\nUsage:")
print(json.dumps(usage, ensure_ascii=False, indent=2))

Round-trip latency (ms): 14201.83
Model response:
```json
{"selected_option": 4, "answer_text": "Liaoning, Shenyang, Begoyoy District, Moscow, Russia", "reason": "Dòng Su-27 được phát triển tiếp bởi Sukhoi (tại Moscow, Nga) và các nhà sản xuất tại Trung Quốc (như Shenyang tại tỉnh Liaoning để phát triển dòng J-11). Vietnam chủ yếu là bên sử dụng chứ không phải nhà sản xuất phát triển dòng model mới từ Su-27."}
```

Usage:
{
  "completion_tokens": 111,
  "prompt_tokens": 247,
  "total_tokens": 358,
  "completion_tokens_details": {
    "accepted_prediction_tokens": null,
    "audio_tokens": 0,
    "reasoning_tokens": 0,
    "rejected_prediction_tokens": null,
    "image_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 0,
    "cache_write_tokens": 0,
    "video_tokens": 0
  },
  "cost": 7.404e-05,
  "is_byok": false,
  "cost_details": {
    "upstream_inference_cost": 7.404e-05,
    "upstream_inference_prompt_cost": 2.964e-05,
    "upstream_inference

## 5. Parse, chấm thử và lưu log cho 1 mẫu

Kết quả được lưu vào `results/method_1_1_smoke/{run_id}/` gồm JSONL và CSV một dòng. Các cột quan trọng cho thống kê sau này là `question_type`, `input_tokens`, `output_tokens`, `total_tokens`, `cost`, `round_trip_latency_ms`, `is_correct`.

In [10]:
def parse_selected_option(text: str) -> int | None:
    cleaned = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", cleaned, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        cleaned = fenced.group(1).strip()

    try:
        payload = json.loads(cleaned)
        selected = payload.get("selected_option")
        return int(selected) if selected is not None else None
    except Exception:
        match = re.search(r"selected_option[^0-9]*(\d+)", cleaned)
        return int(match.group(1)) if match else None


def append_jsonl(path: Path, record: dict) -> None:
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_single_row_csv(path: Path, record: dict) -> None:
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=list(record.keys()))
        writer.writeheader()
        writer.writerow(record)


predicted_option = parse_selected_option(content)
is_correct = predicted_option == sample["correct_option"]
parse_success = predicted_option is not None

result = {
    "run_id": RUN_ID,
    "method_id": METHOD_ID,
    "method_name": METHOD_NAME,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "sample_id": sample["number"],
    "question_type": sample["question_type"],
    "question": sample["question"],
    "gold_answer": sample["gold_answer"],
    "correct_option": sample["correct_option"],
    "predicted_option": predicted_option,
    "is_correct": is_correct,
    "parse_success": parse_success,
    "model": MODEL,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "input_tokens": usage.get("prompt_tokens"),
    "output_tokens": usage.get("completion_tokens"),
    "total_tokens": usage.get("total_tokens"),
    "cost": usage.get("cost"),
    "round_trip_latency_ms": round_trip_latency_ms,
    "raw_response": content,
}

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
append_jsonl(RESULTS_DIR / "method_1_1_sample_result.jsonl", result)
write_single_row_csv(RESULTS_DIR / "method_1_1_sample_summary.csv", result)

print(json.dumps(result, ensure_ascii=False, indent=2))
print(f"\nSaved result to: {RESULTS_DIR}")

{
  "run_id": "method-1-1-smoke-20260613-164159-aafdf3f0",
  "method_id": "1.1",
  "method_name": "LLM-only baseline",
  "timestamp_utc": "2026-06-13T16:42:14.036067+00:00",
  "sample_id": 43,
  "question_type": "multi-hop",
  "question": "Kể tên vị trí của những nhà sản xuất máy bay đã chọn dòng Sukhoi Su-27 để phát triển tiếp các model máy bay khác?",
  "gold_answer": "Liaoning, Shenyang, Begoyoy District, Moscow, Russia",
  "correct_option": 4,
  "predicted_option": 4,
  "is_correct": true,
  "parse_success": true,
  "model": "google/gemma-4-26b-a4b-it",
  "temperature": 0,
  "max_tokens": 256,
  "input_tokens": 247,
  "output_tokens": 111,
  "total_tokens": 358,
  "cost": 7.404e-05,
  "round_trip_latency_ms": 14201.833200000692,
  "raw_response": "```json\n{\"selected_option\": 4, \"answer_text\": \"Liaoning, Shenyang, Begoyoy District, Moscow, Russia\", \"reason\": \"Dòng Su-27 được phát triển tiếp bởi Sukhoi (tại Moscow, Nga) và các nhà sản xuất tại Trung Quốc (như Shenyang tại t